In [14]:
import re
import pandas as pd
import numpy as np
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer

In [2]:
## Load data from pickled files
df = pd.read_pickle("pos_sent.pkl")
df.head()

,GAME_ID,GAME_NAME,COMMENT,RATING,CMT_LEN,CMT_SENT,KEY_PHRASES
1,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,I was favorably surprised by this I didnt buy ...,8.5,792,0.9751,"[consider somewhat mid im, also strikingly wel..."
2,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite in the Pandemic Legacy series and ...,8.5,318,0.9096,"[ive bought another copy, favorite legacy game..."
3,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,My favorite coop game,8.5,23,0.4588,[favorite coop game]
4,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,This is a must buy legacy game for anyone that...,8.5,131,0.5106,"[must buy legacy game, cleanest legacy games, ..."
5,6c485b5a-283e-4f21-98cf-b058efcfe147,Pandemic Legacy: Season 1,First complete play through was awesome played...,8.5,97,0.9186,"[first complete play, still great, campaign 2,..."


In [17]:
## Remove common noice in natural language
def clean_text(text):
    text = text.lower()
    text = re.sub(r'\d+', '', text)
    text = re.sub(r'[^\w\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    
## Create list of documents (comments)
documents = np.array(df['COMMENT'])

cleaned_docs = np.vectorize(clean_text)(documents)
cleaned_docs = cleaned_docs[cleaned_docs != None]

In [23]:
documents

array(['I was favorably surprised by this I didnt buy it when it first came out because I thought it would be gimmicky  change the board as you play But actually it is fantastic  now I see what all the hype was about It takes pandemic which I consider somewhat mid Im not super into vanilla cooperative games and turns it into a roleplaying like experience with twists and turns in the story line Its also strikingly well balanced and well designed even with all of the changes to the rules as the game progresses Your abilities scale with the difficulties quite well My wife and I had a blast Slight knock to the rating because we probably wont play through it a second time but for a number of months it was the only thing we played  thanks to my students for getting it for us',
       'My favorite in the Pandemic Legacy series and frankly my favorite legacy game period This game was full of so many twists turns and surprises While there is a main storyline that plays out the areascountries im

In [18]:
## Create TfIdf vectorizer and fit_transform it to the comments
vectorizer = TfidfVectorizer(
    max_df=0.95,
    min_df=5,
    stop_words='english',
    ngram_range=(1,2)
)
TfIdf_matrix = vectorizer.fit_transform(cleaned_docs)

## Create the NMF Model for topic analysis
model = NMF(n_components=3, random_state=42)
W = model.fit_transform(TfIdf_matrix)
H = model.components_

ValueError: empty vocabulary; perhaps the documents only contain stop words

In [13]:
## Get the top keywords in each topic
feature_names = vectorizer.get_feature_names_out()

for topic_idx, topic in enumerate(H):
    top_words = [feature_names[i] for i in topic.argsort()[:-11:-1]]
    print(f"Topic #{topic_idx + 1}: {' '.join(top_words)}")

Topic #1: game great play great game like fun good played really time
Topic #2: best best game played game game played ive ive played best board best coop game ive
Topic #3: love love game game game love absolutely love love love absolutely just love love theme theme
